In [ ]:
import pandas as pd
 
PLIK_CSV = "wszystkie_miasta_CH4.csv"  # <- podmień na ścieżkę do swojego pliku
 
df = pd.read_csv(PLIK_CSV, low_memory=False)
df["Data"] = pd.to_datetime(df["Data"], errors="coerce")
 
# Kolumny wspólne, które mają się znaleźć w każdej ramce
KOLUMNY_WSPOLNE = [
    "Data",
    "Rok",
    "Miesiac",
    "Sezon",
    "miejsce",
    "lon_pomiaru",
    "lat_pomiaru",
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
]
 
# Mapowanie: przyjazna nazwa gazu -> nazwa kolumny w pliku źródłowym
KOLUMNY_GAZOW = {
    "CH4": "CH4_column_volume_mixing_ratio_dry_air",
    "CO": "CO_column_number_density",
    "HCHO": "tropospheric_HCHO_column_number_density",
    "NO2": "NO2_column_number_density",
    "NO2_tropospheric": "tropospheric_NO2_column_number_density",
    "O3": "O3_column_number_density",
    "SO2": "SO2_column_number_density",
}
 
ramki_gazow = {}
 
print("=" * 70)
print("PODSUMOWANIE DOSTĘPNOŚCI DANYCH DLA POSZCZEGÓLNYCH GAZÓW")
print("=" * 70)
 
for nazwa_gazu, kolumna in KOLUMNY_GAZOW.items():
    if kolumna not in df.columns:
        print(f"{nazwa_gazu}: brak kolumny '{kolumna}' w pliku - pomijam")
        continue
 
    liczba_pomiarow = df[kolumna].notna().sum()
    if liczba_pomiarow == 0:
        print(f"{nazwa_gazu} ({kolumna}): 0 pomiarów - pomijam (pusta kolumna)")
        continue
 
    ramka = df.loc[df[kolumna].notna(), KOLUMNY_WSPOLNE + [kolumna]].copy()
    ramka = ramka.rename(columns={kolumna: "wartosc"})
    ramka = ramka.reset_index(drop=True)
 
    ramki_gazow[nazwa_gazu] = ramka
    print(f"{nazwa_gazu} ({kolumna}): {len(ramka)} wierszy, "
          f"zakres dat {ramka['Data'].min().date()} -- {ramka['Data'].max().date()}")
 
# ---------------------------------------------------------------------------
# Podgląd każdej ramki
# ---------------------------------------------------------------------------
for nazwa_gazu, ramka in ramki_gazow.items():
    print("\n" + "=" * 70)
    print(f"DataFrame dla gazu: {nazwa_gazu}  (kolumna źródłowa: {KOLUMNY_GAZOW[nazwa_gazu]})")
    print("=" * 70)
    print(ramka.head())
    print(f"Kształt: {ramka.shape}")

PODSUMOWANIE DOSTĘPNOŚCI DANYCH DLA POSZCZEGÓLNYCH GAZÓW
CH4 (CH4_column_volume_mixing_ratio_dry_air): 192146 wierszy, zakres dat 2019-01-28 -- 2026-06-30
CO (CO_column_number_density): 1977 wierszy, zakres dat 2021-02-25 -- 2026-03-07
HCHO (tropospheric_HCHO_column_number_density): 4649 wierszy, zakres dat 2019-02-27 -- 2026-03-07
NO2 (NO2_column_number_density): 1142 wierszy, zakres dat 2026-03-06 -- 2026-03-07
NO2_tropospheric (tropospheric_NO2_column_number_density): 315 wierszy, zakres dat 2026-03-06 -- 2026-03-07
O3 (O3_column_number_density): 0 pomiarów - pomijam (pusta kolumna)
SO2 (SO2_column_number_density): 7229 wierszy, zakres dat 2019-06-14 -- 2026-03-16

DataFrame dla gazu: CH4  (kolumna źródłowa: CH4_column_volume_mixing_ratio_dry_air)
        Data   Rok  Miesiac Sezon                  miejsce  lon_pomiaru  \
0 2019-02-05  2019        2  ZIMA  bogdaj-uciechow-czeszow    17.386892   
1 2019-02-05  2019        2  ZIMA  bogdaj-uciechow-czeszow    17.449774   
2 2019-02-05  

# Gotowa funkcja 

In [11]:
"""
Funkcja do analizy trendu czasowego i sezonowości dla dowolnego gazu
=====================================================================

Zawiera jedną, uniwersalną funkcję `analizuj_trend_i_sezonowosc(df, ...)`,
która przyjmuje DataFrame (np. jedną z ramek zwróconych przez
`wyciagnij_gazy.py`, czyli `ramki_gazow["CH4"]`, `ramki_gazow["CO"]` itd.)
i generuje komplet wykresów trendu oraz sezonowości.

Wymagane biblioteki: pandas, numpy, matplotlib, statsmodels
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf


def analizuj_trend_i_sezonowosc(
    df: pd.DataFrame,
    kolumna_wartosci: str = "wartosc",
    kolumna_daty: str = "Data",
    kolumna_sezon: str = "Sezon",
    kolumna_temperatura: str = "temperature_2m_mean",
    nazwa_gazu: str = "gaz",
    folder_wyjsciowy: str = ".",
    okres_dekompozycji: int = 365,
    pokaz_wykresy: bool = False,
) -> dict:
    """
    Przeprowadza pełną analizę trendu czasowego i sezonowości dla wskazanej
    kolumny wartości w DataFrame i zapisuje komplet wykresów jako pliki PNG.

    Parametry
    ---------
    df : pd.DataFrame
        Dane wejściowe, np. `ramki_gazow["CH4"]`.
    kolumna_wartosci : str
        Nazwa kolumny z wartością mierzonego gazu.
    kolumna_daty : str
        Nazwa kolumny z datą pomiaru.
    kolumna_sezon : str
        Nazwa kolumny z porą roku (opcjonalna - jeśli brak, wykres pomijany).
    kolumna_temperatura : str
        Nazwa kolumny ze średnią temperaturą (opcjonalna).
    nazwa_gazu : str
        Nazwa używana w tytułach wykresów i nazwach plików.
    folder_wyjsciowy : str
        Folder, do którego zapisywane są wykresy PNG.
    okres_dekompozycji : int
        Okres sezonowości (w dniach) używany do dekompozycji szeregu (domyślnie 365).
    pokaz_wykresy : bool
        Jeśli True, dodatkowo wyświetla wykresy (plt.show()) - przydatne w Jupyterze.

    Zwraca
    ------
    dict
        Słownik z podsumowaniem liczbowym analizy (m.in. nachylenie trendu,
        miesiąc/max/min, ścieżki zapisanych wykresów) oraz z pomocniczymi
        zagregowanymi seriami (dzienne/miesięczne/roczne).
    """

    os.makedirs(folder_wyjsciowy, exist_ok=True)
    prefiks = os.path.join(folder_wyjsciowy, nazwa_gazu.replace(" ", "_"))
    zapisane_wykresy = []

    def zapisz(fig, sufiks):
        sciezka = f"{prefiks}_{sufiks}.png"
        fig.tight_layout()
        fig.savefig(sciezka, dpi=150)
        if not pokaz_wykresy:
            plt.close(fig)
        zapisane_wykresy.append(sciezka)
        return sciezka

    # -----------------------------------------------------------------
    # 0. Przygotowanie danych
    # -----------------------------------------------------------------
    dane = df.copy()
    dane[kolumna_daty] = pd.to_datetime(dane[kolumna_daty], errors="coerce")
    dane = dane.dropna(subset=[kolumna_daty, kolumna_wartosci])
    dane = dane.sort_values(kolumna_daty)

    if dane.empty:
        raise ValueError("Brak danych do analizy po usunięciu braków.")

    dane["Miesiac_num"] = dane[kolumna_daty].dt.month
    dane["Rok_num"] = dane[kolumna_daty].dt.year

    # Agregacja: 1 wartość na dzień (średnia ze wszystkich pomiarów danego dnia)
    dzienne = dane.groupby(kolumna_daty)[kolumna_wartosci].mean().to_frame("wartosc")
    miesieczne = dzienne["wartosc"].resample("MS").mean()
    roczne = dzienne["wartosc"].resample("YS").mean()

    print("=" * 70)
    print(f"ANALIZA TRENDU I SEZONOWOŚCI: {nazwa_gazu}")
    print("=" * 70)
    print(f"Liczba obserwacji: {len(dane)}")
    print(f"Zakres dat: {dane[kolumna_daty].min().date()} -- {dane[kolumna_daty].max().date()}")
    print(f"Średnia wartość: {dane[kolumna_wartosci].mean():.6g}")
    print(f"Odchylenie standardowe: {dane[kolumna_wartosci].std():.6g}")

    # -----------------------------------------------------------------
    # WYKRES 1: szereg czasowy (dzienny + miesięczny) + linia trendu liniowego
    # -----------------------------------------------------------------
    miesieczne_bez_na = miesieczne.dropna()
    nachylenie_mies = np.nan
    if len(miesieczne_bez_na) > 1:
        x_num = np.arange(len(miesieczne_bez_na))
        wspolczynniki = np.polyfit(x_num, miesieczne_bez_na.values, 1)
        linia_trendu = np.polyval(wspolczynniki, x_num)
        nachylenie_mies = wspolczynniki[0]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(dzienne.index, dzienne["wartosc"], ".", alpha=0.25, color="gray", label="Pomiary dzienne")
    ax.plot(miesieczne.index, miesieczne.values, "-o", color="tab:blue", label="Średnia miesięczna")
    if len(miesieczne_bez_na) > 1:
        ax.plot(miesieczne_bez_na.index, linia_trendu, "--", color="tab:red", linewidth=2, label="Linia trendu")
    ax.set_title(f"{nazwa_gazu}: trend czasowy")
    ax.set_xlabel("Data")
    ax.set_ylabel(kolumna_wartosci)
    ax.legend()
    ax.grid(alpha=0.3)
    zapisz(fig, "01_trend_szereg_czasowy")

    # -----------------------------------------------------------------
    # WYKRES 2: średnia krocząca (wygładzony trend, 30-dniowa)
    # -----------------------------------------------------------------
    srednia_kroczaca = dzienne["wartosc"].rolling(window=30, min_periods=5).mean()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(dzienne.index, dzienne["wartosc"], ".", alpha=0.2, color="gray", label="Pomiary dzienne")
    ax.plot(srednia_kroczaca.index, srednia_kroczaca.values, color="tab:green", linewidth=2,
            label="Średnia krocząca (30 dni)")
    ax.set_title(f"{nazwa_gazu}: wygładzony trend (średnia krocząca)")
    ax.set_xlabel("Data")
    ax.set_ylabel(kolumna_wartosci)
    ax.legend()
    ax.grid(alpha=0.3)
    zapisz(fig, "02_srednia_kroczaca")

    # -----------------------------------------------------------------
    # WYKRES 3: średnie roczne (słupkowy)
    # -----------------------------------------------------------------
    roczne_bez_na = roczne.dropna()
    if len(roczne_bez_na) > 0:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.bar(roczne_bez_na.index.year.astype(str), roczne_bez_na.values, color="tab:blue")
        ax.set_title(f"{nazwa_gazu}: średnia wartość wg roku")
        ax.set_xlabel("Rok")
        ax.set_ylabel(kolumna_wartosci)
        ax.grid(alpha=0.3, axis="y")
        zapisz(fig, "03_srednia_roczna")

    # -----------------------------------------------------------------
    # WYKRES 4: sezonowość - boxplot wg miesiąca
    # -----------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 5))
    dane_boxplot = [dane.loc[dane["Miesiac_num"] == m, kolumna_wartosci].dropna() for m in range(1, 13)]
    ax.boxplot(dane_boxplot, tick_labels=[str(m) for m in range(1, 13)])
    ax.set_title(f"{nazwa_gazu}: sezonowość - rozkład wg miesiąca")
    ax.set_xlabel("Miesiąc")
    ax.set_ylabel(kolumna_wartosci)
    ax.grid(alpha=0.3)
    zapisz(fig, "04_sezonowosc_boxplot_miesiac")

    # -----------------------------------------------------------------
    # WYKRES 5: sezonowość - średnia + odchylenie wg miesiąca (linia)
    # -----------------------------------------------------------------
    srednia_wg_miesiaca = dane.groupby("Miesiac_num")[kolumna_wartosci].agg(["mean", "std"])
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.errorbar(
        srednia_wg_miesiaca.index, srednia_wg_miesiaca["mean"],
        yerr=srednia_wg_miesiaca["std"], marker="o", capsize=4, color="tab:purple"
    )
    ax.set_title(f"{nazwa_gazu}: średnia (+/- odch. std) wg miesiąca")
    ax.set_xlabel("Miesiąc")
    ax.set_ylabel(kolumna_wartosci)
    ax.set_xticks(range(1, 13))
    ax.grid(alpha=0.3)
    zapisz(fig, "05_sezonowosc_srednia_miesiac")

    # -----------------------------------------------------------------
    # WYKRES 6: sezonowość wg pory roku (Sezon) - jeśli kolumna istnieje
    # -----------------------------------------------------------------
    if kolumna_sezon in dane.columns and dane[kolumna_sezon].notna().any():
        srednia_wg_sezonu = dane.groupby(kolumna_sezon)[kolumna_wartosci].mean().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.bar(srednia_wg_sezonu.index.astype(str), srednia_wg_sezonu.values, color="tab:orange")
        ax.set_title(f"{nazwa_gazu}: średnia wartość wg pory roku")
        ax.set_xlabel("Sezon")
        ax.set_ylabel(kolumna_wartosci)
        ax.grid(alpha=0.3, axis="y")
        zapisz(fig, "06_sezonowosc_pora_roku")

    # -----------------------------------------------------------------
    # WYKRES 7: heatmapa rok x miesiąc (wizualizacja sezonowości + trendu razem)
    # -----------------------------------------------------------------
    tabela_przestawna = dane.pivot_table(
        index="Rok_num", columns="Miesiac_num", values=kolumna_wartosci, aggfunc="mean"
    )
    if tabela_przestawna.shape[0] > 1 and tabela_przestawna.shape[1] > 1:
        fig, ax = plt.subplots(figsize=(10, max(3, 0.5 * tabela_przestawna.shape[0] + 2)))
        im = ax.imshow(tabela_przestawna.values, aspect="auto", cmap="viridis")
        ax.set_xticks(range(tabela_przestawna.shape[1]))
        ax.set_xticklabels(tabela_przestawna.columns)
        ax.set_yticks(range(tabela_przestawna.shape[0]))
        ax.set_yticklabels(tabela_przestawna.index)
        ax.set_xlabel("Miesiąc")
        ax.set_ylabel("Rok")
        ax.set_title(f"{nazwa_gazu}: heatmapa średnich wartości (rok x miesiąc)")
        fig.colorbar(im, ax=ax, label=kolumna_wartosci)
        zapisz(fig, "07_heatmapa_rok_miesiac")

    # -----------------------------------------------------------------
    # WYKRES 8: histogram / rozkład wartości
    # -----------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(dane[kolumna_wartosci].dropna(), bins=40, color="tab:cyan", edgecolor="black", alpha=0.8)
    ax.set_title(f"{nazwa_gazu}: rozkład wartości")
    ax.set_xlabel(kolumna_wartosci)
    ax.set_ylabel("Liczba obserwacji")
    ax.grid(alpha=0.3)
    zapisz(fig, "08_histogram")

    # -----------------------------------------------------------------
    # WYKRES 9: zależność od temperatury (jeśli dostępna)
    # -----------------------------------------------------------------
    if kolumna_temperatura in dane.columns and dane[kolumna_temperatura].notna().any():
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(dane[kolumna_temperatura], dane[kolumna_wartosci], alpha=0.3, s=10, color="tab:red")
        korelacja = dane[[kolumna_temperatura, kolumna_wartosci]].corr().iloc[0, 1]
        ax.set_title(f"{nazwa_gazu} vs temperatura (korelacja r = {korelacja:.3f})")
        ax.set_xlabel(kolumna_temperatura)
        ax.set_ylabel(kolumna_wartosci)
        ax.grid(alpha=0.3)
        zapisz(fig, "09_vs_temperatura")
    else:
        korelacja = np.nan

    # -----------------------------------------------------------------
    # WYKRES 10: dekompozycja szeregu (trend / sezonowość / reszty)
    # -----------------------------------------------------------------
    szereg_pelny = dzienne["wartosc"].asfreq("D").interpolate(method="linear")
    dekompozycja_ok = len(szereg_pelny) >= 2 * okres_dekompozycji
    if dekompozycja_ok:
        dekompozycja = seasonal_decompose(szereg_pelny, model="additive", period=okres_dekompozycji)
        fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
        dekompozycja.observed.plot(ax=axes[0], title="Dane obserwowane")
        dekompozycja.trend.plot(ax=axes[1], title="Trend")
        dekompozycja.seasonal.plot(ax=axes[2], title="Sezonowość")
        dekompozycja.resid.plot(ax=axes[3], title="Reszty (szum)")
        zapisz(fig, "10_dekompozycja")
    else:
        print(
            f"Uwaga: pominięto dekompozycję (potrzeba >= {2 * okres_dekompozycji} dni "
            f"ciągłego zakresu, dostępne: {len(szereg_pelny)})."
        )

    # -----------------------------------------------------------------
    # WYKRES 11: autokorelacja (ACF) - czy są powtarzalne wzorce czasowe
    # -----------------------------------------------------------------
    if len(szereg_pelny.dropna()) > 20:
        fig, ax = plt.subplots(figsize=(10, 4))
        plot_acf(szereg_pelny.dropna(), ax=ax, lags=min(365, len(szereg_pelny.dropna()) // 2))
        ax.set_title(f"{nazwa_gazu}: autokorelacja (ACF)")
        zapisz(fig, "11_autokorelacja")

    # -----------------------------------------------------------------
    # Podsumowanie
    # -----------------------------------------------------------------
    if not srednia_wg_miesiaca.empty:
        miesiac_max = srednia_wg_miesiaca["mean"].idxmax()
        miesiac_min = srednia_wg_miesiaca["mean"].idxmin()
    else:
        miesiac_max = miesiac_min = None

    podsumowanie = {
        "nazwa_gazu": nazwa_gazu,
        "liczba_obserwacji": len(dane),
        "data_min": dane[kolumna_daty].min(),
        "data_max": dane[kolumna_daty].max(),
        "srednia": dane[kolumna_wartosci].mean(),
        "odchylenie_std": dane[kolumna_wartosci].std(),
        "nachylenie_trendu_miesiecznego": nachylenie_mies,
        "miesiac_najwyzsza_srednia": miesiac_max,
        "miesiac_najnizsza_srednia": miesiac_min,
        "korelacja_z_temperatura": korelacja,
        "dekompozycja_wykonana": dekompozycja_ok,
        "zapisane_wykresy": zapisane_wykresy,
        "dzienne": dzienne,
        "miesieczne": miesieczne,
        "roczne": roczne,
    }

    print(f"\nZapisano {len(zapisane_wykresy)} wykresów w folderze '{folder_wyjsciowy}':")
    for sciezka in zapisane_wykresy:
        print("  -", sciezka)

    return podsumowanie



# Podstawowa analiza - CH4

In [26]:
wynik = analizuj_trend_i_sezonowosc(ramki_gazow["CH4"], nazwa_gazu="CH4", folder_wyjsciowy="wykresy_CH4")
podsumowanie = test_istotnosci_trendu(wynik, uzyj="miesieczne")

ANALIZA TRENDU I SEZONOWOŚCI: CH4
Liczba obserwacji: 192146
Zakres dat: 2019-01-28 -- 2026-06-30
Średnia wartość: 1878.59
Odchylenie standardowe: 26.1427

Zapisano 11 wykresów w folderze 'wykresy_CH4':
  - wykresy_CH4\CH4_01_trend_szereg_czasowy.png
  - wykresy_CH4\CH4_02_srednia_kroczaca.png
  - wykresy_CH4\CH4_03_srednia_roczna.png
  - wykresy_CH4\CH4_04_sezonowosc_boxplot_miesiac.png
  - wykresy_CH4\CH4_05_sezonowosc_srednia_miesiac.png
  - wykresy_CH4\CH4_06_sezonowosc_pora_roku.png
  - wykresy_CH4\CH4_07_heatmapa_rok_miesiac.png
  - wykresy_CH4\CH4_08_histogram.png
  - wykresy_CH4\CH4_09_vs_temperatura.png
  - wykresy_CH4\CH4_10_dekompozycja.png
  - wykresy_CH4\CH4_11_autokorelacja.png
TEST ISTOTNOŚCI TRENDU: CH4  (seria: miesieczne, n=44)

[1] Klasyczna regresja liniowa (scipy.stats.linregress)
    nachylenie (slope)     = 1.50129 / okres
    p-value                = 1.056e-16
    R^2                    = 0.8092
    -> trend ISTOTNY statystycznie przy alpha=0.05 (UWAGA: zakłada b

In [25]:
wynik = analizuj_trend_i_sezonowosc(ramki_gazow["CO"], nazwa_gazu="CO", folder_wyjsciowy="wykresy_CO")
podsumowanie = test_istotnosci_trendu(wynik, uzyj="miesieczne")

ANALIZA TRENDU I SEZONOWOŚCI: CO
Liczba obserwacji: 1977
Zakres dat: 2021-02-25 -- 2026-03-07
Średnia wartość: 0.0350557
Odchylenie standardowe: 0.00268158

Zapisano 11 wykresów w folderze 'wykresy_CO':
  - wykresy_CO\CO_01_trend_szereg_czasowy.png
  - wykresy_CO\CO_02_srednia_kroczaca.png
  - wykresy_CO\CO_03_srednia_roczna.png
  - wykresy_CO\CO_04_sezonowosc_boxplot_miesiac.png
  - wykresy_CO\CO_05_sezonowosc_srednia_miesiac.png
  - wykresy_CO\CO_06_sezonowosc_pora_roku.png
  - wykresy_CO\CO_07_heatmapa_rok_miesiac.png
  - wykresy_CO\CO_08_histogram.png
  - wykresy_CO\CO_09_vs_temperatura.png
  - wykresy_CO\CO_10_dekompozycja.png
  - wykresy_CO\CO_11_autokorelacja.png
TEST ISTOTNOŚCI TRENDU: CO  (seria: miesieczne, n=6)

[1] Klasyczna regresja liniowa (scipy.stats.linregress)
    nachylenie (slope)     = -0.00011281 / okres
    p-value                = 0.8375
    R^2                    = 0.0118
    -> trend NIEISTOTNY statystycznie przy alpha=0.05 (UWAGA: zakłada brak autokorelacji r

In [24]:
wynik = analizuj_trend_i_sezonowosc(ramki_gazow["NO2_tropospheric"], nazwa_gazu="NO2_tropospheric", folder_wyjsciowy="wykresy_NO2_tropospheric")
podsumowanie = test_istotnosci_trendu(wynik, uzyj="dzienne")

ANALIZA TRENDU I SEZONOWOŚCI: NO2_tropospheric
Liczba obserwacji: 315
Zakres dat: 2026-03-06 -- 2026-03-07
Średnia wartość: 6.43122e-05
Odchylenie standardowe: 2.69296e-05
Uwaga: pominięto dekompozycję (potrzeba >= 730 dni ciągłego zakresu, dostępne: 2).

Zapisano 8 wykresów w folderze 'wykresy_NO2_tropospheric':
  - wykresy_NO2_tropospheric\NO2_tropospheric_01_trend_szereg_czasowy.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_02_srednia_kroczaca.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_03_srednia_roczna.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_04_sezonowosc_boxplot_miesiac.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_05_sezonowosc_srednia_miesiac.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_06_sezonowosc_pora_roku.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_08_histogram.png
  - wykresy_NO2_tropospheric\NO2_tropospheric_09_vs_temperatura.png


ValueError: Za mało punktów (2) w serii 'dzienne', by sensownie testować istotność trendu. Spróbuj innej agregacji (np. 'dzienne').

In [23]:
wynik = analizuj_trend_i_sezonowosc(ramki_gazow["NO2"], nazwa_gazu="NO2", folder_wyjsciowy="wykresy_NO2")
podsumowanie = test_istotnosci_trendu(wynik, uzyj="dzienne")

ANALIZA TRENDU I SEZONOWOŚCI: NO2
Liczba obserwacji: 1142
Zakres dat: 2026-03-06 -- 2026-03-07
Średnia wartość: 7.46739e-05
Odchylenie standardowe: 2.38968e-05
Uwaga: pominięto dekompozycję (potrzeba >= 730 dni ciągłego zakresu, dostępne: 2).

Zapisano 8 wykresów w folderze 'wykresy_NO2':
  - wykresy_NO2\NO2_01_trend_szereg_czasowy.png
  - wykresy_NO2\NO2_02_srednia_kroczaca.png
  - wykresy_NO2\NO2_03_srednia_roczna.png
  - wykresy_NO2\NO2_04_sezonowosc_boxplot_miesiac.png
  - wykresy_NO2\NO2_05_sezonowosc_srednia_miesiac.png
  - wykresy_NO2\NO2_06_sezonowosc_pora_roku.png
  - wykresy_NO2\NO2_08_histogram.png
  - wykresy_NO2\NO2_09_vs_temperatura.png


ValueError: Za mało punktów (2) w serii 'dzienne', by sensownie testować istotność trendu. Spróbuj innej agregacji (np. 'dzienne').

In [15]:
wynik = analizuj_trend_i_sezonowosc(ramki_gazow["SO2"], nazwa_gazu="SO2", folder_wyjsciowy="wykresy_SO2")

# istotnosc statystyczna?? halo??
podsumowanie = test_istotnosci_trendu(wynik, uzyj="miesieczne")


ANALIZA TRENDU I SEZONOWOŚCI: SO2
Liczba obserwacji: 7229
Zakres dat: 2019-06-14 -- 2026-03-16
Średnia wartość: 0.000386485
Odchylenie standardowe: 0.000808157

Zapisano 11 wykresów w folderze 'wykresy_SO2':
  - wykresy_SO2\SO2_01_trend_szereg_czasowy.png
  - wykresy_SO2\SO2_02_srednia_kroczaca.png
  - wykresy_SO2\SO2_03_srednia_roczna.png
  - wykresy_SO2\SO2_04_sezonowosc_boxplot_miesiac.png
  - wykresy_SO2\SO2_05_sezonowosc_srednia_miesiac.png
  - wykresy_SO2\SO2_06_sezonowosc_pora_roku.png
  - wykresy_SO2\SO2_07_heatmapa_rok_miesiac.png
  - wykresy_SO2\SO2_08_histogram.png
  - wykresy_SO2\SO2_09_vs_temperatura.png
  - wykresy_SO2\SO2_10_dekompozycja.png
  - wykresy_SO2\SO2_11_autokorelacja.png
TEST ISTOTNOŚCI TRENDU: SO2  (seria: miesieczne, n=16)

[1] Klasyczna regresja liniowa (scipy.stats.linregress)
    nachylenie (slope)     = 8.01201e-06 / okres
    p-value                = 0.657
    R^2                    = 0.0145
    -> trend NIEISTOTNY statystycznie przy alpha=0.05 (UWAGA: 

In [17]:
wynik = analizuj_trend_i_sezonowosc(ramki_gazow["HCHO"], nazwa_gazu="HCHO", folder_wyjsciowy="wykresy_HCHO")
podsumowanie = test_istotnosci_trendu(wynik, uzyj="miesieczne")

ANALIZA TRENDU I SEZONOWOŚCI: HCHO
Liczba obserwacji: 4649
Zakres dat: 2019-02-27 -- 2026-03-07
Średnia wartość: 7.51134e-05
Odchylenie standardowe: 0.000128511

Zapisano 11 wykresów w folderze 'wykresy_HCHO':
  - wykresy_HCHO\HCHO_01_trend_szereg_czasowy.png
  - wykresy_HCHO\HCHO_02_srednia_kroczaca.png
  - wykresy_HCHO\HCHO_03_srednia_roczna.png
  - wykresy_HCHO\HCHO_04_sezonowosc_boxplot_miesiac.png
  - wykresy_HCHO\HCHO_05_sezonowosc_srednia_miesiac.png
  - wykresy_HCHO\HCHO_06_sezonowosc_pora_roku.png
  - wykresy_HCHO\HCHO_07_heatmapa_rok_miesiac.png
  - wykresy_HCHO\HCHO_08_histogram.png
  - wykresy_HCHO\HCHO_09_vs_temperatura.png
  - wykresy_HCHO\HCHO_10_dekompozycja.png
  - wykresy_HCHO\HCHO_11_autokorelacja.png
TEST ISTOTNOŚCI TRENDU: HCHO  (seria: miesieczne, n=13)

[1] Klasyczna regresja liniowa (scipy.stats.linregress)
    nachylenie (slope)     = 3.26871e-06 / okres
    p-value                = 0.5228
    R^2                    = 0.0381
    -> trend NIEISTOTNY statystyczni

In [20]:
"""
Test istotności statystycznej trendu czasowego
================================================

Funkcja `test_istotnosci_trendu(wynik, ...)` przyjmuje słownik zwrócony przez
`analizuj_trend_i_sezonowosc()` (czyli zmienną `wynik` z poprzedniego kroku)
i sprawdza, czy widoczny trend jest statystycznie istotny, trzema metodami:

1. Klasyczna regresja liniowa (scipy.stats.linregress) - najprostszy test,
   ale ZAKŁADA niezależne, nieskorelowane reszty. Dane atmosferyczne
   (jak Twoje CH4/CO) są silnie autoskorelowane w czasie (widać to na
   wykresie ACF), więc ten test zwykle zawyża istotność (p-value wychodzi
   sztucznie niskie / "zbyt pewne").

2. Regresja liniowa z odpornymi błędami standardowymi HAC/Newey-West
   (statsmodels OLS, cov_type="HAC") - poprawia problem z punktu 1,
   uwzględniając autokorelację reszt. To bardziej wiarygodny p-value.

3. Test Manna-Kendalla (pymannkendall) - nieparametryczny test monotoniczności
   trendu, standardowy w naukach o atmosferze/klimacie (np. do trendów CO2,
   CH4, temperatury). Nie zakłada normalności ani liniowości, jest odporny
   na wartości odstające. Dodatkowo liczy odporny estymator nachylenia
   (Sen's slope / Theil-Sen), mniej wrażliwy na outliery niż zwykłe OLS.

Wymagane biblioteki: numpy, pandas, scipy, statsmodels, pymannkendall
    pip install pymannkendall
"""

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import pymannkendall as mk


def test_istotnosci_trendu(
    wynik: dict,
    uzyj: str = "miesieczne",
    alpha: float = 0.05,
) -> dict:
    """
    Testuje istotność statystyczną trendu na podstawie wyniku funkcji
    `analizuj_trend_i_sezonowosc()`.

    Parametry
    ---------
    wynik : dict
        Słownik zwrócony przez `analizuj_trend_i_sezonowosc()`, zawierający
        m.in. zagregowane serie 'dzienne', 'miesieczne', 'roczne'.
    uzyj : str
        Która seria ma być testowana: "dzienne", "miesieczne" (zalecane -
        mniej autokorelacji niż dane dzienne) albo "roczne".
    alpha : float
        Poziom istotności (domyślnie 0.05).

    Zwraca
    ------
    dict z wynikami trzech testów oraz czytelnym podsumowaniem.
    """

    seria = wynik[uzyj].dropna()
    nazwa_gazu = wynik.get("nazwa_gazu", "gaz")

    if len(seria) < 4:
        raise ValueError(
            f"Za mało punktów ({len(seria)}) w serii '{uzyj}', by sensownie "
            "testować istotność trendu. Spróbuj innej agregacji (np. 'dzienne')."
        )

    y = seria.values.astype(float)
    x = np.arange(len(y))  # kolejne okresy (miesiące/dni/lata) jako liczby 0,1,2,...

    print("=" * 70)
    print(f"TEST ISTOTNOŚCI TRENDU: {nazwa_gazu}  (seria: {uzyj}, n={len(y)})")
    print("=" * 70)

    # -----------------------------------------------------------------
    # 1. Klasyczna regresja liniowa (OLS, zwykłe błędy standardowe)
    # -----------------------------------------------------------------
    reg = stats.linregress(x, y)
    istotny_ols = reg.pvalue < alpha
    print("\n[1] Klasyczna regresja liniowa (scipy.stats.linregress)")
    print(f"    nachylenie (slope)     = {reg.slope:.6g} / okres")
    print(f"    p-value                = {reg.pvalue:.4g}")
    print(f"    R^2                    = {reg.rvalue**2:.4f}")
    print(f"    -> trend {'ISTOTNY' if istotny_ols else 'NIEISTOTNY'} statystycznie "
          f"przy alpha={alpha} (UWAGA: zakłada brak autokorelacji reszt)")

    # -----------------------------------------------------------------
    # 2. OLS z odpornymi błędami standardowymi HAC (Newey-West)
    #    - koryguje zawyżoną istotność spowodowaną autokorelacją
    # -----------------------------------------------------------------
    X = sm.add_constant(x)
    # maxlags dobrany "z rozsądku" - ok. sqrt(n), typowa heurystyka Newey-West
    maxlags = max(1, int(np.sqrt(len(y))))
    model_hac = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    slope_hac = model_hac.params[1]
    p_hac = model_hac.pvalues[1]
    istotny_hac = p_hac < alpha
    print("\n[2] Regresja liniowa z błędami HAC/Newey-West (odporna na autokorelację)")
    print(f"    nachylenie (slope)     = {slope_hac:.6g} / okres")
    print(f"    p-value (HAC)          = {p_hac:.4g}")
    print(f"    -> trend {'ISTOTNY' if istotny_hac else 'NIEISTOTNY'} statystycznie "
          f"przy alpha={alpha} (bardziej wiarygodne niż [1] dla danych autoskorelowanych)")

    # -----------------------------------------------------------------
    # 3. Test Manna-Kendalla + odporne nachylenie Sena (Theil-Sen)
    # -----------------------------------------------------------------
    wynik_mk = mk.original_test(y, alpha=alpha)
    istotny_mk = wynik_mk.p < alpha
    print("\n[3] Test Manna-Kendalla (nieparametryczny, standard w klimatologii)")
    print(f"    trend                   = {wynik_mk.trend}  (increasing / decreasing / no trend)")
    print(f"    p-value                 = {wynik_mk.p:.4g}")
    print(f"    Tau Kendalla             = {wynik_mk.Tau:.4f}")
    print(f"    nachylenie Sena (Sen's slope) = {wynik_mk.slope:.6g} / okres")
    print(f"    -> trend {'ISTOTNY' if istotny_mk else 'NIEISTOTNY'} statystycznie przy alpha={alpha}")

    # -----------------------------------------------------------------
    # Podsumowanie
    # -----------------------------------------------------------------
    liczba_zgodnych = sum([istotny_ols, istotny_hac, istotny_mk])
    print("\n" + "-" * 70)
    print(f"PODSUMOWANIE: {liczba_zgodnych}/3 testy wskazują na istotny statystycznie trend.")
    if istotny_hac and istotny_mk:
        print("Wniosek: trend jest wiarygodnie istotny statystycznie "
              "(potwierdzają to zarówno test odporny na autokorelację, jak i test nieparametryczny).")
    elif not istotny_hac and not istotny_mk:
        print("Wniosek: brak wystarczających dowodów na istotny statystycznie trend "
              "po uwzględnieniu autokorelacji danych.")
    else:
        print("Wniosek: wyniki niejednoznaczne - testy [2] i [3] nie są zgodne, "
              "warto przeanalizować dane dokładniej (np. długość szeregu, wartości odstające).")
    print("-" * 70)

    return {
        "seria_uzyta": uzyj,
        "n": len(y),
        "ols": {"slope": reg.slope, "p_value": reg.pvalue, "r2": reg.rvalue**2, "istotny": istotny_ols},
        "ols_hac": {"slope": slope_hac, "p_value": p_hac, "maxlags": maxlags, "istotny": istotny_hac},
        "mann_kendall": {
            "trend": wynik_mk.trend,
            "p_value": wynik_mk.p,
            "tau": wynik_mk.Tau,
            "sen_slope": wynik_mk.slope,
            "istotny": istotny_mk,
        },
    }


51139
